In [1]:
!pip install -q rouge-score sacrebleu transformers sentencepiece

from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.7 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import torch, os, json
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration


MBART_PATH = "/content/drive/MyDrive/cultural-recipes-data/mbart-checkpoint-19830"


tokenizer = MBart50TokenizerFast.from_pretrained(MBART_PATH)

model = MBartForConditionalGeneration.from_pretrained(MBART_PATH)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")



Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
#model evaluation (after successful final training)

model.eval()

def adapt_recipe(recipe_text, max_length=256):
    inputs = tokenizer(
        recipe_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=4
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

en_recipe = (
    "Title: Scrambled Eggs, "
    "Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper, "
    "Steps: Crack eggs into bowl. Whisk well. "
    "Melt butter in pan over medium heat. "
    "Add eggs and stir gently until cooked."
)

cn_recipe = (
    "Title: 番茄炒蛋, "
    "Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, "
    "Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，"
    "加入番茄翻炒，加盐和糖调味即可。"
)

print("=== TEST 1: English → Chinese ===")
print("INPUT: ", en_recipe[:80], "...")
print("OUTPUT:", adapt_recipe(en_recipe))

print("\n=== TEST 2: Chinese → English ===")
print("INPUT: ", cn_recipe[:80], "...")
print("OUTPUT:", adapt_recipe(cn_recipe))

=== TEST 1: English → Chinese ===
INPUT:  Title: Scrambled Eggs, Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper,  ...
OUTPUT: Title: Scrambled Eggs, Ingredients: 4 eggs, 1 Tbsp. butter, salt and pepper to taste, Steps: Melt butter in skillet. Add eggs and scramble. Season with salt and pepper.

=== TEST 2: Chinese → English ===
INPUT:  Title: 番茄炒蛋, Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，加入 ...
OUTPUT: Title: Deviled Eggs With Tomatoes, Ingredients: 6 hard-cooked eggs, cut lengthwise in half, 1/4 cup KRAFT Real Mayo Mayonnaise, 2 Tbsp. CLAUSSEN Sweet Pickle Relish, 2 tsp. GREY POUPON Dijon Mustard, Steps: Mash egg yolks in medium bowl with fork. Add mayo, relish and mustard; mix well. Fill whites with yolk mixture.


In [ ]:
#DO NOT RUN THIS ~ bad decoding
model.eval()

def generate_mbart(source_text, src_lang, tgt_lang, max_length=256):
    """Generate adaptation using mBART with explicit language tokens."""
    tokenizer.src_lang = src_lang

    inputs = tokenizer(
        source_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)

    # Force target language token
    forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

en_recipe = (
    "Title: Scrambled Eggs, "
    "Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper, "
    "Steps: Crack eggs into bowl. Whisk well. "
    "Melt butter in pan. Add eggs and stir until cooked."
)

cn_recipe = (
    "Title: 番茄炒蛋, "
    "Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, "
    "Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，"
    "加入番茄翻炒，加盐和糖调味即可。"
)

print("=== TEST 1: English → Chinese ===")
print("INPUT: ", en_recipe[:80], "...")
print("OUTPUT:", generate_mbart(en_recipe, src_lang="en_XX", tgt_lang="zh_CN"))

print("\n=== TEST 2: Chinese → English ===")
print("INPUT: ", cn_recipe[:80], "...")
print("OUTPUT:", generate_mbart(cn_recipe, src_lang="zh_CN", tgt_lang="en_XX"))

=== TEST 1: English → Chinese ===
INPUT:  Title: Scrambled Eggs, Ingredients: 3 eggs, 2 tablespoons butter, salt, pepper,  ...
OUTPUT: Title: Scrambled Eggs, Ingredients: 4 eggs, 1 Tbsp. butter, salt and pepper to taste, Steps: Melt butter in skillet. Add eggs and scramble. Season with salt and pepper.

=== TEST 2: Chinese → English ===
INPUT:  Title: 番茄炒蛋, Ingredients: 3个鸡蛋, 2个番茄, 盐, 糖, 食用油, Steps: 鸡蛋打散。番茄切块。热锅加油，炒鸡蛋至凝固，加入 ...
OUTPUT: Title: 西红柿炒鸡蛋, Ingredients: 2个西红柿, 2个鸡蛋, 适量葱花, 适量盐, 适量鸡精, 适量油, Steps: 西红柿切块,鸡蛋打散,葱花切好备用。 锅中放油,油热后放入葱花爆香。 放入西红柿翻炒。 西红柿炒出汁后,放入鸡蛋。 鸡蛋炒熟后,放入适量盐,鸡精,出锅。


In [ ]:
#Evaluation on human/silver sets

from rouge_score import rouge_scorer
import sacrebleu

BASE = "/content/drive/MyDrive/cultural-recipes-data"

EVAL_SETS = {
    "human_cn2en":  (f"{BASE}/human_cn2en/sampled_human_mbart_cn2en.jsonl",  "zh_CN", "en_XX", False),
    "human_en2cn":  (f"{BASE}/human_en2cn/sampled_human_mbart_en2cn.jsonl",  "en_XX", "zh_CN", True),
    "silver_cn2en": (f"{BASE}/silver_cn2en/sampled_silver_mbart_cn2en.jsonl", "zh_CN", "en_XX", False),
    "silver_en2cn": (f"{BASE}/silver_en2cn/sampled_silver_mbart_en2cn.jsonl", "en_XX", "zh_CN", True),
}

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def compute_metrics(predictions, references, is_chinese_target=False):
    bleu = sacrebleu.corpus_bleu(
        predictions, [references],
        tokenize="char" if is_chinese_target else "13a"
    )
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    r1, r2, rL = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)
    return {
        "bleu":   round(bleu.score, 2),
        "rouge1": round(sum(r1)/len(r1)*100, 2),
        "rouge2": round(sum(r2)/len(r2)*100, 2),
        "rougeL": round(sum(rL)/len(rL)*100, 2),
    }

mbart_results = {}

for name, (path, src_lang, tgt_lang, is_chinese) in EVAL_SETS.items():
    print(f"\n=== {name} ===")
    records = load_jsonl(path)
    print(f"Records: {len(records)}")

    our_preds, baseline_preds, references = [], [], []

    for i, record in enumerate(records):
        references.append(record["references"][0])
        baseline_preds.append(record["prediction"])
        our_preds.append(
            adapt_recipe(record["source"])
        )
        if (i+1) % 5 == 0:
            print(f"  Generated {i+1}/{len(records)}...")

    our_scores      = compute_metrics(our_preds, references, is_chinese)
    baseline_scores = compute_metrics(baseline_preds, references, is_chinese)
    mbart_results[name] = {
        "our_scores": our_scores,
        "baseline_scores": baseline_scores
    }

    print(f"\n  {'Metric':<10} {'mBART(partial)':>15} {'Paper mT5-base':>15} {'Diff':>10}")
    print(f"  {'-'*52}")
    for metric in ["bleu", "rouge1", "rouge2", "rougeL"]:
        ours  = our_scores[metric]
        paper = baseline_scores[metric]
        diff  = ours - paper
        sign  = "+" if diff > 0 else ""
        print(f"  {metric:<10} {ours:>15.2f} {paper:>15.2f} {sign}{diff:>9.2f}")


=== human_cn2en ===
Records: 25
  Generated 5/25...
  Generated 10/25...
  Generated 15/25...
  Generated 20/25...
  Generated 25/25...

  Metric      mBART(partial)  Paper mT5-base       Diff
  ----------------------------------------------------
  bleu                  4.30            5.05     -0.75
  rouge1               28.23           28.89     -0.66
  rouge2                5.41            4.94 +     0.47
  rougeL               20.70           19.74 +     0.96

=== human_en2cn ===
Records: 41
  Generated 5/41...
  Generated 10/41...
  Generated 15/41...
  Generated 20/41...
  Generated 25/41...
  Generated 30/41...
  Generated 35/41...
  Generated 40/41...

  Metric      mBART(partial)  Paper mT5-base       Diff
  ----------------------------------------------------
  bleu                  7.07           16.43     -9.36
  rouge1               20.35           32.73    -12.38
  rouge2                3.11           12.51     -9.40
  rougeL               16.68           30.54    -13.